<a href="https://colab.research.google.com/github/Rais-Ataullov/BigData/blob/lab1/L1_Apache_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-8-jdk-headless > /dev/null

!wget -q https://archive.apache.org/dist/spark/spark-3.4.0/spark-3.4.0-bin-hadoop3.tgz
!tar -xf spark-3.4.0-bin-hadoop3.tgz

!pip install -q pyspark==3.4.0

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.0-bin-hadoop3"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

!wget https://archive.apache.org/dist/hadoop/common/hadoop-3.3.4/hadoop-3.3.4.tar.gz
!tar -xzf hadoop-3.3.4.tar.gz
!mv hadoop-3.3.4 /opt/hadoop

os.environ["HADOOP_HOME"] = "/opt/hadoop"
os.environ["PATH"] += ":/opt/hadoop/bin:/opt/hadoop/sbin"

# Создание Resilient Distributed Dataset

In [31]:
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
import numpy as np

conf = SparkConf().setAppName("L1_Apache_Spark").setMaster('local[*]')

sc = SparkContext(conf=conf)
spark = SparkSession(sc)

Создайте RDD для текстового файла warandpeace.txt.

_Примечание_. При наборе команд используйте TAB - функцию автодополнения.

In [32]:
warandpeace = sc.textFile("warandsociety.txt")

В данной команде указывается относительный путь, который начинается с вашей папки в РФС.

Выведите количество строк файла.

In [4]:
warandpeace.count()

12851

In [33]:
nilFile = sc.textFile("nil")
nilFile.count()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/content/nil
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:304)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:244)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:332)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:208)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:291)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:287)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:291)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:287)
	at org.apache.spark.api.python.PythonRDD.getPartitions(P

Заметьте, что первая команда выполняется успешно, а вторая выводит сообщение, что такого файла нет. Это происходит потому, что выполнение обработки в Spark является ленивым и не запускается, до встречи команды действия(action). count - первая команда действия, с которой вы познакомились.

Считайте первые 10 строк файла warandsociety.txt.

In [5]:
warandpeace.take(10)

['Лев Николаевич Толстой',
 'Война и мир. Книга 1',
 '',
 'Война и мир – 1',
 '',
 ' ',
 ' http://www.lib.ru',
 '',
 'Аннотация ',
 '']

Создайте распределённую коллекцию из нескольких элементов и для каждого элемента верните ip адрес, на котором он расположен:

In [6]:
import socket

hostnames = sc.parallelize([1, 2, 3]).map(lambda x: socket.gethostname()).collect()
print(hostnames)

['210d23d1c3df', '210d23d1c3df', '210d23d1c3df']


# Обработка текста

Найдите строки, в которых содержится слово "война".

In [7]:
linesWithWar = warandpeace.filter(lambda x: "война" in x)
print(linesWithWar.count())

54


_Примечание_. Аргументом filter является лямбда функция - функция без имени. До обозначения => в скобках через запятую следуют переменные аргументов функции, затем следует команда языка Scala. При использовании фигурных скобок язык позволяет описывать лямбда функции с цепочкой команд в теле, аналогично именованным функциям.

Запросите первую строку. Строкой в данном файле является целый абзац, так как только по завершению абзаца содержится символ переноса строки.

In [8]:
linesWithWar.first()

"– Еh bien, mon prince. Genes et Lucques ne sont plus que des apanages, des поместья, de la famille Buonaparte. Non, je vous previens, que si vous ne me dites pas, que nous avons la guerre, si vous vous permettez encore de pallier toutes les infamies, toutes les atrocites de cet Antichrist (ma parole, j'y crois) – je ne vous connais plus, vous n'etes plus mon ami, vous n'etes plus мой верный раб, comme vous dites. [Ну, что, князь, Генуа и Лукка стали не больше, как поместьями фамилии Бонапарте. Нет, я вас предупреждаю, если вы мне не скажете, что у нас война, если вы еще позволите себе защищать все гадости, все ужасы этого Антихриста (право, я верю, что он Антихрист) – я вас больше не знаю, вы уж не друг мой, вы уж не мой верный раб, как вы говорите.] Ну, здравствуйте, здравствуйте. Je vois que je vous fais peur, [Я вижу, что я вас пугаю,] садитесь и рассказывайте."

Данные могут быть перемещены в кэш. Этот приём очень полезен при повторном обращении к данным, для запросов "горячих" данных или запуска итеративных алгоритмов.

Перед подсчётом количества элементов вызовите команду кэширования cache(). Трансформации не будут обработаны, пока не будет запущена одна из команд - действий.

Воспользуйтесь следующим блоком кода для замера времени выполнения команды.

In [9]:
def time(f):
    import time
    t = time.process_time()
    f()
    print(f"Elapsed time: {int((time.process_time() - t)*1e9)} ns")

linesWithWar.cache()
time(lambda: linesWithWar.count() )
time(lambda: linesWithWar.count() )

Elapsed time: 5627583 ns
Elapsed time: 6447986 ns


При выполнении команды count второй раз вы должны заметить небольшое ускорение. Кэширование небольших файлов не даёт большого преимущества, однако для огромных файлов, распределённых по сотням или тысячам узлов, разница во времени выполнения может быть существенной. Вторая команда linesWithWar.count() выполняется над результатом от предшествующих команде cache трансформаций и на больших объёмах данных будет ускорять выполнение последующих команд.

Найдите гистограмму слов:

In [10]:
wordCounts = linesWithWar.flatMap(lambda line: line.split(" ")).map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)
wordCounts.saveAsTextFile("warandpeace_histogram.txt")

Сохраните результаты в файл, а затем, найдите данные в HDFS и выведите данные в linux консоли с помощью команды hadoop fs -cat warandpeace_histogram.txt/* (здесь используется относительный путь).

In [11]:
! hadoop fs -cat warandpeace_histogram.txt/*

('Еh', 1)
('prince.', 1)
('et', 4)
('sont', 1)
('plus', 3)
('des', 2)
('la', 6)
('Buonaparte.', 1)
('Non,', 2)
('vous', 10)
('me', 1)
('nous', 1)
('guerre,', 2)
('permettez', 1)
('encore', 1)
('toutes', 2)
('les', 3)
('infamies,', 1)
('Antichrist', 1)
('(ma', 1)
('parole,', 1)
('connais', 1)
('мой', 3)
('comme', 2)
('dites.', 1)
('[Ну,', 1)
('князь,', 3)
('поместьями', 1)
('фамилии', 1)
('Бонапарте.', 1)
('Нет,', 4)
('если', 2)
('скажете,', 1)
('у', 12)
('война,', 10)
('еще', 8)
('гадости,', 1)
('ужасы', 1)
('(право,', 1)
('верю,', 1)
('он', 55)
('Антихрист)', 1)
('знаю,', 3)
('уж', 3)
('мой,', 1)
('Ну,', 2)
('здравствуйте,', 1)
('vois', 1)
('fais', 1)
('[Я', 2)
('вижу,', 3)
('пугаю,]', 1)
('садитесь', 1)
('mari', 1)
("m'abandonne,", 1)
('продолжала', 1)
('она', 7)
('обращаясь', 1)
('генералу,', 1)
('il', 1)
('tuer.', 1)
('moi,', 1)
('покидает', 1)
('меня.', 1)
('Идет', 1)
('на', 56)
('Скажите,', 1)
('гадкая', 1)
('война,]', 1)
('князю', 1)
('ответа,', 1)
('князя', 7)
('Василия,', 1)
(

In [12]:
import re
p = re.compile('\w+')
letters = p.findall("a b c")
[print(l) for l in letters]

a
b
c


<>:2: SyntaxWarning: invalid escape sequence '\w'
<>:2: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_1702/1728730318.py:2: SyntaxWarning: invalid escape sequence '\w'
  p = re.compile('\w+')


[None, None, None]

## Упражнение. Улучшите процедуру, убирая из слов лишние символы и трансформируя все слова в нижний регистр. Используйте регулярные выражения.

In [13]:
def clean_word_advanced(word):
    word = word.lower()
    word = re.sub(r'^[^\w\'-]+|[^\w\'-]+$', '', word)
    word = re.sub(r'[^a-zа-яё0-9\'-]', '', word)
    return word

def clean_word_with_accents(word):
    word = word.lower()
    replacements = { 'é': 'e', 'è': 'e', 'ê': 'e', 'ë': 'e', 'à': 'a', 'â': 'a', 'ä': 'a',
                    'î': 'i', 'ï': 'i', 'ô': 'o', 'ö': 'o', 'ù': 'u', 'û': 'u', 'ü': 'u', 'ç': 'c'
    }
    for accented, plain in replacements.items():
        word = word.replace(accented, plain)
    word = re.sub(r'[^a-zа-яё0-9\'-]', '', word)
    return word

wordCounts_upgraded = linesWithWar.flatMap(lambda line: line.split()).map(lambda word: (clean_word_advanced(word), 1)) \
    .filter(lambda pair: len(pair[0]) > 1).reduceByKey(lambda a, b: a + b)

wordCounts_upgraded.saveAsTextFile("warandpeace_histogram_upgraded.txt")
! hadoop fs -cat warandpeace_histogram_upgraded.txt/*

('genes', 1)
('et', 4)
('sont', 1)
('plus', 5)
('des', 2)
('поместья', 1)
('la', 7)
('non', 2)
('vous', 11)
('me', 1)
('pas', 1)
('nous', 1)
('permettez', 1)
('encore', 1)
('toutes', 2)
('les', 3)
('infamies', 1)
('antichrist', 1)
('ma', 1)
('parole', 1)
('connais', 1)
('мой', 4)
('comme', 2)
('генуа', 1)
('поместьями', 1)
('фамилии', 1)
('бонапарте', 4)
('предупреждаю', 1)
('если', 2)
('война', 55)
('еще', 10)
('ужасы', 1)
('он', 75)
('антихрист', 1)
('знаю', 3)
('уж', 3)
('vois', 1)
('fais', 1)
('садитесь', 1)
('рассказывайте', 1)
('mari', 1)
('продолжала', 1)
('она', 8)
('обращаясь', 1)
('il', 1)
('moi', 2)
('покидает', 1)
('меня', 9)
('идет', 2)
('на', 59)
('смерть', 2)
('скажите', 2)
('гадкая', 1)
('князю', 1)
('князя', 7)
('элен', 2)
('был', 15)
('вот', 3)
('голову', 1)
('хотел', 5)
('теперь', 15)
('ежели', 15)
('это', 37)
('была', 27)
('бы', 21)
('понял', 2)
('первый', 4)
('англии', 1)
('человека', 8)
('нехорошо', 1)
('да', 5)
('ведь', 2)
('говорят', 7)
('объявлена', 2)
('гостья

# Операции с множествами

Инициализируйте два множества

In [14]:
a = sc.parallelize([1,2,3,4])
b = sc.parallelize([3,4,6,7])

Найдите объединение a и b и соберите данные на главный узел с помощью функции collect.

In [15]:
a.union(b).collect()

[1, 2, 3, 4, 3, 4, 6, 7]

Обратите внимание, что общие элементы дублируются, поэтому результат не является классическим множеством на самом деле. Такое поведение делает это операцию очень дешёвой, так как обновляется только информация о местонахождении данных для данного RDD. Уберите дубликаты с помощью distinct.

In [16]:
a.union(b).distinct().collect()

[4, 1, 2, 6, 3, 7]

Найдите пересечение множеств.

In [17]:
a.intersection(b).collect()

[4, 3]

Найдите разность множеств.

In [18]:
a.subtract(b).collect()

[1, 2]

## Упражнение. Найдите в исходном коде Spark определение функции distinct. Объясните почему реализация этой операции действительно убирает дубликаты.

Пример: rdd = [1, 2, 2, 3, 3, 3, 4]

Шаг 1: map(x => (x, null)) Превращаем каждый элемент в пару (ключ, значение) Результат: [(1, None), (2, None), (2, None), (3, None), (3, None), (3, None), (4, None)]

Шаг 2: reduceByKey((a, b) => a) Группируем по ключу и оставляем только первое значение Ключ 1: [(1, None)] -> (1, None) Ключ 2: [(2, None), (2, None)] -> (2, None) # берем первое Ключ 3: [(3, None), (3, None), (3, None)] -> (3, None) Ключ 4: [(4, None)] -> (4, None) Результат: [(1, None), (2, None), (3, None), (4, None)]

Шаг 3: map(_._1) Извлекаем только ключ (первый элемент пары) Результат: [1, 2, 3, 4]

# Общие переменные

В Apache Spark общими переменными являются широковещательные (broadcast) переменные и аккумулирующие (accumulator) переменные.

### Широковещательные переменные

Общие переменные удобны если вы обращаетесь к небольшому объёму данных на всех узлах. Например, это могут быть параметры алгоритмов обработки, небольшие матрицы.

В консоли, с которой вы работали в предыдущем разделе, создайте широковещательную переменную.

In [35]:
broadcastVar = sc.broadcast([1,2,3])

Для получения значения обратитесь к полю value:

In [36]:
broadcastVar.value

[1, 2, 3]

### Аккумулирующие переменные

Аккумулирующие переменные являются объектами, которые могут быть изменены только ассоциативной операцией добавления. Они используются для эффективной реализации счётчиков и суммарных значений. Вы можете также использовать свой тип, над котором определена ассоциативная операция при необходимости.

Особенностью использования переменной является возможность доступа к значению только на узле в driver процессе.

Потренируйтесь в создании аккумулирующих переменных:

In [20]:
accum = sc.accumulator(0)

10

Следующим шагом запустите параллельную обработку массива и в каждом параллельном задании добавьте к аккумулирующей переменной значение элемента массива:

In [ ]:
sc.parallelize([1,2,3,4]).foreach(lambda x: accum.add(x))

Для получения текущего значения вызовите команду:

In [ ]:
accum.value

Результатом должно быть число 10. Пары ключ-значение Создайте пару ключ-значение из двух букв:

In [37]:
pair = ('a', 'b')

Для доступа к первому значению обратитесь к полю _1:

In [ ]:
print(pair[0])

Для доступа к второму значению к полю _2:

In [ ]:
print(pair[1])

Если распределённая коллекция состоит из пар, то они трактуются как для ключ-значение и для таких коллекций доступны дополнительные операции. Наиболее распространённые, это: группировка по ключу, агрегирование значений с одинаковыми ключами, объединение двух коллекций по ключу.

# Топ-10 популярных номеров такси

Проанализируем данные о поездках такси в Нью-Йорке и найдём 10 номеров такси, которые совершили наибольшее количество поездок.

В первую очередь будет необходимо загрузить данные в MapR-FS. Создайте новую папку в MapR-FS:

Создайте RDD на основе загруженных данных nyctaxi.csv:

In [38]:
taxi = sc.textFile("nyctaxi.csv")

Выведите первые 5 строк из данной таблицы:

In [39]:
for t in taxi.take(5):
    print(t)

"_id","_rev","dropoff_datetime","dropoff_latitude","dropoff_longitude","hack_license","medallion","passenger_count","pickup_datetime","pickup_latitude","pickup_longitude","rate_code","store_and_fwd_flag","trip_distance","trip_time_in_secs","vendor_id"
"29b3f4a30dea6688d4c289c9672cb996","1-ddfdec8050c7ef4dc694eeeda6c4625e","2013-01-11 22:03:00",+4.07033460000000E+001,-7.40144200000000E+001,"A93D1F7F8998FFB75EEF477EB6077516","68BC16A99E915E44ADA7E639B4DD5F59",2,"2013-01-11 21:48:00",+4.06760670000000E+001,-7.39810790000000E+001,1,,+4.08000000000000E+000,900,"VTS"
"2a80cfaa425dcec0861e02ae44354500","1-b72234b58a7b0018a1ec5d2ea0797e32","2013-01-11 04:28:00",+4.08190960000000E+001,-7.39467470000000E+001,"64CE1B03FDE343BB8DFB512123A525A4","60150AA39B2F654ED6F0C3AF8174A48A",1,"2013-01-11 04:07:00",+4.07280540000000E+001,-7.40020370000000E+001,1,,+8.53000000000000E+000,1260,"VTS"
"29b3f4a30dea6688d4c289c96758d87e","1-387ec30eac5abda89d2abefdf947b2c1","2013-01-11 22:02:00",+4.07277180000000E+00

Обратите внимание, что первая строка является заголовком. Её как правило нужно будет отфильтровать. Одним из эффективных способов является следующий:

In [40]:
import itertools
taxi.mapPartitionsWithIndex(lambda idx, it:  itertools.islice(it,1,None) if (idx==0) else it  )

PythonRDD[8] at RDD at PythonRDD.scala:53

_Примечание_. Для анализа структурированных табличных данных рассматривайте в качестве альтернативы использование SQL API и DataSet API.

Для разбора значений потребуется создать RDD, где каждая строка разбита на массив подстрок. Используйте запятую в качестве разделителя. Наберите:

In [23]:
taxiParse = taxi.map(lambda line: line.split(","))

Теперь преобразуем массив строк в массив пар ключ-значение, где ключом будет служить номер такси (6 колонка), а значением единица.

In [24]:
taxiMedKey = taxiParse.map(lambda row: (row[6], 1))
for t in taxiMedKey.take(5):
    print(t)

('"medallion"', 1)
('"68BC16A99E915E44ADA7E639B4DD5F59"', 1)
('"60150AA39B2F654ED6F0C3AF8174A48A"', 1)
('"6F907BC9A85B7034C8418A24A0A75489"', 1)
('"1AFFD48CC07161DA651625B562FE4D06"', 1)


Следом мы можем найти количество поездок каждого номера такси:

In [25]:
taxiMedCounts = taxiMedKey.reduceByKey(lambda v1, v2: v1+v2)

Выведем полученные результаты в отсортированном виде:

In [26]:
top10 = taxiMedCounts.map(lambda x: x[::-1]).top(10)
for x in top10:
    print(x[::-1])

('"AB44AD9A03B7CFAF3925103BDCC0AF23"', 33)
('"B30AAC3A6BAB4417C3BE8DA726E422A4"', 32)
('"874B24DA5769D8B22F4136FF55361C45"', 32)
('"7181737C1274E597B4C054D185748A62"', 31)
('"EA4F7251D970B0B69E7C6F0C20752409"', 30)
('"73495B48481E673AD8D7578764C9EC98"', 30)
('"71CACFBADF9568AAE88A843DB511D172"', 30)
('"443668D5FC95733983627585D1233DA8"', 30)
('"D32F83E5B54623D4ECF0FED59FD706CF"', 29)
('"71A81E9114851674595A13481B5AC886"', 29)


_Примечание_. Нотация _.swap является объявлением анонимной функции от одного аргумента, аналог записи x => x.swap.

Являются ли обе map операции распределёнными? Найдите в документации Spark в классах RDD или PairRDDFunctions метод top.

Вы также можете сгруппировать все описанные выше трансформации, преобразующие исходные данные в одну цепочку:

In [27]:
taxiCounts = taxi.map(lambda line: line.split(",")).map(lambda row: (row[6],1)).reduceByKey(lambda a,b: a + b)

_Примечание_. Нотация _ + _ является объявлением анонимной функции от двух аргументов, аналог более многословной записи (a,b) => a + b.

Попробуйте найти общее количество номеров такси несколько раз, предварительно объявив RDD taxiCounts как сохраняемую в кэше:

In [28]:
top10 = taxiMedCounts.map(lambda x: x[::-1]).top(10)
for x in top10:
    print(x[::-1])

('"AB44AD9A03B7CFAF3925103BDCC0AF23"', 33)
('"B30AAC3A6BAB4417C3BE8DA726E422A4"', 32)
('"874B24DA5769D8B22F4136FF55361C45"', 32)
('"7181737C1274E597B4C054D185748A62"', 31)
('"EA4F7251D970B0B69E7C6F0C20752409"', 30)
('"73495B48481E673AD8D7578764C9EC98"', 30)
('"71CACFBADF9568AAE88A843DB511D172"', 30)
('"443668D5FC95733983627585D1233DA8"', 30)
('"D32F83E5B54623D4ECF0FED59FD706CF"', 29)
('"71A81E9114851674595A13481B5AC886"', 29)


Сравните время, которое трансформации выполняются первый раз и второй. Чем больше данные, тем существеннее разница.

In [29]:
taxiCounts.cache()
time(lambda: taxiCounts.count())
time(lambda: taxiCounts.count())

Elapsed time: 5072504 ns
Elapsed time: 5887752 ns


In [30]:
sc.stop()